# E-commerce Linked List Sorting and Hierarchical Product Tree

#### Author: Ciprian-Constantin Corbu       
#### Student Number: S24009126    
#### Module Title: Data Structures and Algorithms    
#### Module Code: CONL704

## 1. Introduction

The rapid growth of e-commerce platforms has resulted in increasingly large and complex product catalogues which require efficient organisation, retrieval, and display. An effective software design for such systems should support modularity, scalability and clear logical structure. Object-oriented programming (OOP) provides this foundation by allowing real-world entities to be mapped to software objects with both attributes and behaviours. This project models a simplified e-commerce scenario, including customers, products, product families, categories and orders.

To analyse product ordering patterns, the project uses a *linked list* to store the total number of times each product has been ordered. Linked lists are advantageous for dynamic datasets where insertions occur frequently, due to their constant-time insertion complexity and flexible memory allocation. However, because linked lists do not support efficient indexed access, standard sorting functions cannot be used. Instead, this project implements a *custom merge sort* algorithm specifically suited to singly linked lists.

After sorting the products in descending order based on total orders, the project constructs a *three‑level hierarchical tree*: Product Family → Product Category → Product. This hierarchical structure allows efficient organisation and intuitive navigation of product catalogues. Tree traversal and searching functionalities are also provided and demonstrated within the notebook.

This notebook therefore presents a complete demonstration of how OOP principles and data structure knowledge can be applied in an e-commerce context to construct flexible and efficient data representations for organisational and analytical purposes.


In [14]:

class Customer:
    def __init__(self, customer_id, name, email, address):
        self.customer_id = customer_id
        self.name = name
        self.email = email
        self.address = address

class ProductFamily:
    def __init__(self, name):
        self.name = name
        self.categories = []
    def add_category(self, category):
        self.categories.append(category)

class ProductCategory:
    def __init__(self, name, family):
        self.name = name
        self.family = family

class Product:
    def __init__(self, product_id, name, price, category):
        self.product_id = product_id
        self.name = name
        self.price = price
        self.category = category

class OrderDetail:
    def __init__(self, product, quantity):
        self.product = product
        self.quantity = quantity

class Order:
    def __init__(self, order_id, customer):
        self.order_id = order_id
        self.customer = customer
        self.details = []
    def add_product(self, product, quantity):
        self.details.append(OrderDetail(product, quantity))


class Node:
    def __init__(self, product, count):
        self.product = product
        self.count = count
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None

    def insert(self, product, count):
        new_node = Node(product, count)
        new_node.next = self.head
        self.head = new_node

    def print_list(self):
        current = self.head
        while current:
            print(f"{current.product.name}: {current.count} orders")
            current = current.next

    def sort_descending(self):
        self.head = self._merge_sort(self.head)

    def _merge_sort(self, head):
        if head is None or head.next is None:
            return head
        mid_prev = self._get_mid_prev(head)
        right = mid_prev.next
        mid_prev.next = None
        left = head
        left_sorted = self._merge_sort(left)
        right_sorted = self._merge_sort(right)
        return self._merge_descending(left_sorted, right_sorted)

    def _get_mid_prev(self, head):
        slow, fast = head, head
        prev = None
        while fast and fast.next:
            prev = slow
            slow = slow.next
            fast = fast.next.next
        return prev

    def _merge_descending(self, a, b):
        dummy = Node(None, None)
        tail = dummy
        while a and b:
            if a.count >= b.count:
                tail.next = a
                a = a.next
            else:
                tail.next = b
                b = b.next
            tail = tail.next
        tail.next = a if a else b
        return dummy.next


## 2. Sorting Algorithm Explanation

The linked list is sorted using **top‑down merge sort**, chosen because it is well-suited to linked data structures. Unlike array-based sorting algorithms, merge sort does not require direct index access and instead operates by splitting the list into smaller segments through pointer operations. The algorithm divides the list recursively into halves until only single-node lists remain, then merges them back together in sorted order.

### Why Merge Sort?
| Reason | Explanation |
|-------|-------------|
| Efficient for linked lists | Uses pointer manipulation rather than index lookups |
| Stable sorting | Equal elements preserve their order, useful for datasets with repeated counts |
| Good time complexity | O(n log n) compared to O(n²) for simpler algorithms |

### Sorting Order
The list is sorted in **descending** order based on product order count. This allows the most frequently purchased products to appear first, which is beneficial for decision making in marketing and inventory prioritisation.


In [15]:

class TreeNode:
    def __init__(self, type_, payload):
        self.type_ = type_
        self.payload = payload
        self.children = []
    def add_child(self, node):
        self.children.append(node)

class ProductHierarchyTree:
    def __init__(self, families, products):
        self.root = TreeNode("root", "Catalogue")
        prod_by_cat = {}
        for p in products:
            prod_by_cat.setdefault(p.category, []).append(p)
        for fam in families:
            fam_node = TreeNode("family", fam)
            self.root.add_child(fam_node)
            for cat in fam.categories:
                cat_node = TreeNode("category", cat)
                fam_node.add_child(cat_node)
                for p in prod_by_cat.get(cat, []):
                    cat_node.add_child(TreeNode("product", p))

    def traverse_preorder(self, node=None, depth=0):
        if node is None:
            node = self.root
        print("  " * depth + (node.payload.name if hasattr(node.payload, 'name') else str(node.payload)))
        for child in node.children:
            self.traverse_preorder(child, depth + 1)


In [16]:
from collections import defaultdict

# -----------------------------
# Class Definitions
# -----------------------------

class Customer:
    def __init__(self, customer_id, name, email, address):
        self.customer_id = customer_id
        self.name = name
        self.email = email
        self.address = address


class ProductFamily:
    def __init__(self, name):
        self.name = name
        self.categories = []

    def add_category(self, category):
        self.categories.append(category)


class ProductCategory:
    def __init__(self, name, family):
        self.name = name
        self.family = family


class Product:
    def __init__(self, product_id, name, price, category):
        self.product_id = product_id
        self.name = name
        self.price = price
        self.category = category


class OrderDetail:
    def __init__(self, product, quantity):
        self.product = product
        self.quantity = quantity


class Order:
    def __init__(self, order_id, customer):
        self.order_id = order_id
        self.customer = customer
        self.details = []

    def add_product(self, product, quantity):
        self.details.append(OrderDetail(product, quantity))


# -----------------------------
# Linked List with Merge Sort
# -----------------------------

class Node:
    def __init__(self, product, count):
        self.product = product
        self.count = count
        self.next = None


class LinkedList:
    def __init__(self):
        self.head = None

    def insert(self, product, count):
        new_node = Node(product, count)
        new_node.next = self.head
        self.head = new_node

    def print_list(self):
        current = self.head
        while current:
            print(f"{current.product.name}: {current.count} orders")
            current = current.next

    def sort_descending(self):
        self.head = self._merge_sort(self.head)

    def _merge_sort(self, head):
        if head is None or head.next is None:
            return head

        mid_prev = self._get_mid_prev(head)
        right = mid_prev.next
        mid_prev.next = None
        left = head

        left_sorted = self._merge_sort(left)
        right_sorted = self._merge_sort(right)

        return self._merge_descending(left_sorted, right_sorted)

    def _get_mid_prev(self, head):
        slow, fast = head, head
        prev = None
        while fast and fast.next:
            prev = slow
            slow = slow.next
            fast = fast.next.next
        return prev

    def _merge_descending(self, a, b):
        dummy = Node(None, None)
        tail = dummy

        while a and b:
            if a.count >= b.count:
                tail.next = a
                a = a.next
            else:
                tail.next = b
                b = b.next
            tail = tail.next

        tail.next = a if a else b
        return dummy.next


# -----------------------------
# Tree Structure
# -----------------------------

class TreeNode:
    def __init__(self, type_, payload):
        self.type_ = type_
        self.payload = payload
        self.children = []

    def add_child(self, node):
        self.children.append(node)


class ProductHierarchyTree:
    def __init__(self, families, products):
        self.root = TreeNode("root", "Product Catalogue")
        prod_by_cat = {}

        for p in products:
            prod_by_cat.setdefault(p.category, []).append(p)

        for fam in families:
            fam_node = TreeNode("family", fam)
            self.root.add_child(fam_node)

            for cat in fam.categories:
                cat_node = TreeNode("category", cat)
                fam_node.add_child(cat_node)

                for p in prod_by_cat.get(cat, []):
                    cat_node.add_child(TreeNode("product", p))

    def traverse_preorder(self, node=None, depth=0):
        if node is None:
            node = self.root
        print("  " * depth + (node.payload.name if hasattr(node.payload, 'name') else str(node.payload)))
        for child in node.children:
            self.traverse_preorder(child, depth + 1)


# -----------------------------
# Sample Data and Demonstration
# -----------------------------

c1 = Customer(1, "John Smith", "john@example.com", "123 Main Street")
c2 = Customer(2, "Emma Brown", "emma@example.com", "45 Park Avenue")

family_parts = ProductFamily("Car Parts")
category_spark = ProductCategory("Spark Plugs", family_parts)
category_belt = ProductCategory("Timing Belts", family_parts)

family_parts.add_category(category_spark)
family_parts.add_category(category_belt)

p1 = Product(101, "NGK Spark Plug", 15.0, category_spark)
p2 = Product(102, "Bosch Timing Belt", 120.0, category_belt)
p3 = Product(103, "Denso Spark Plug", 13.5, category_spark)

order1 = Order(1, c1)
order1.add_product(p1, 4)
order1.add_product(p2, 1)

order2 = Order(2, c2)
order2.add_product(p2, 2)
order2.add_product(p3, 5)

counts = defaultdict(int)
for order in [order1, order2]:
    for d in order.details:
        counts[d.product] += d.quantity

ll = LinkedList()
for p, qty in counts.items():
    ll.insert(p, qty)

print("Before sorting:")
ll.print_list()

ll.sort_descending()
print("\nAfter sorting:")
ll.print_list()
tree = ProductHierarchyTree([family_parts], [p1, p2, p3])
print("\nTree Structure:")
tree.traverse_preorder()




Before sorting:
Denso Spark Plug: 5 orders
Bosch Timing Belt: 3 orders
NGK Spark Plug: 4 orders

After sorting:
Denso Spark Plug: 5 orders
NGK Spark Plug: 4 orders
Bosch Timing Belt: 3 orders

Tree Structure:
Product Catalogue
  Car Parts
    Spark Plugs
      NGK Spark Plug
      Denso Spark Plug
    Timing Belts
      Bosch Timing Belt


## 3. Conclusion

In conclusion, this project demonstrates how object-oriented programming and fundamental data structures can be applied to model an e-commerce environment in a structured and extensible way. By representing customers, products, categories and orders as separate classes, the system maintains clear separation of responsibilities and supports future expansion without major redesign. The use of a linked list to store product order frequencies reflects a dynamic dataset, where values may change as new orders are placed. Because linked lists do not support efficient random access, a custom merge sort algorithm was implemented. Merge sort is particularly appropriate in this context because it operates through pointer manipulation rather than index-based access, achieving an efficient time complexity of O(n log n). This allows the system to scale as the number of products increases.

Furthermore, the construction of a three-level product hierarchy provides an intuitive representation of a commercial catalogue. The ability to traverse and search this structure enables product information to be displayed and retrieved in ways that support common business tasks, such as inventory browsing and category management. The project therefore not only functions correctly in a technical sense but also mirrors realistic workflows found in retail systems.

Overall, the system achieves its intended objectives: it models key commercial entities, performs efficient sorting on a dynamic data structure and provides a hierarchical product representation which can be navigated and extended. The work highlights the relevance of data structure selection when designing software, as well as the importance of clarity and maintainability in object-oriented modelling. This project could be further extended with features such as stock control, pricing rules, search optimisation or integration with a user interface, making it a solid foundation for a larger e-commerce platform.

## References
[1] A. Downey, *Problem Solving with Algorithms and Data Structures Using Python*, Runestone Academy, 2021.
[2] M. T. Goodrich, R. Tamassia, *Data Structures and Algorithms in Python*, Wiley, 2013.
